### `Powershell Commands`

In [ ]:
.\test_lab\testlab\Scripts\activate
& "$env:LOCALAPPDATA\Programs\nu\bin\nu.exe"

### `nvidia-smi`

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute device active: {device}")

# <span style="background-color:green; color:black">FSM</span>
* <span style="color:red">This is red text.</span>

Math: $(Q, \Sigma, \delta, q_0, F)$.



### `TRUTH TABLE`

In [ ]:
from pyeda.inter import exprvars, truthtable, espresso_exprs

a, b, c = exprvars('a', 3)  # Creates a[2]=a, a[1]=b, a[0]=c
outputs = "".join(["1" if i in [0, 2, 4, 5, 6] else "0" for i in range(1 << 3)])
truthtable([c, b, a], outputs)

### `DECIMAL`

In [ ]:
a=1; b=0; c=1
abc = (a << 2) | (b << 1) | c

### `BINARY`

In [ ]:
number=4; bits=8
binary = ""
for i in range(bits-1, -1, -1):
    bit = (number >> i) & 1  
    binary += str(bit)

print(f"{number:08b}")

### `LOGARITHM`

### `COMPLEX #`

### `COMBINATIONAL LOGIC`

In [ ]:
from sympy.logic import SOPform
from sympy import symbols


simplified_logic = SOPform(symbols(['a', 'b', 'c']), minterms=[0, 2, 4, 5, 6], dontcares = [])
sv_expression = str(simplified_logic)
sv_design = f"""
// ---------------------------------------------
// .sv DESIGN FILE
// ---------------------------------------------
module {"module_name"} (
    input logic {', '.join(['a', 'b', 'c'])},
    output logic y_out
);

    assign y_out = {sv_expression};

endmodule
// ---------------------------------------------
"""



# <span style="background-color:green; color:black">MLPerf Benchmark Suite</span>

### `EXECUTION TIME Decorator`

In [ ]:
def _measure_time(func, *args, **kwargs):
    start_time = time.time()
    result = func(*args, **kwargs)
    end_time = time.time()
    return end_time - start_time


### `MEMORY & BANDWIDTH Constructor`
* via Scaled Dot-Product Attention
* int8 is slower that float32 because <span style="color:red">sgemm</span> library for 32bit mat mult. has AVX instructions that process many nos. simult. + <span style="color:red">scaled_score is computed in float</span> and then converted

In [ ]:
seq_len=512; d_model=1024
Q = (np.random.randn(seq_len, d_model) * 10).astype(np.float32)
K = (np.random.randn(seq_len, d_model) * 10).astype(np.float32)
score_matrix = np.matmul(Q, K.T) 
scaled_score = score_matrix / np.sqrt(d_model) # divide by std.dev/sigma so that e^3sigma doesnt overflow
attention_map = np.exp(scaled_score)    # softmax?

### `COMPUTE SPEED (FLOPs) Constructor`
* via Large Dense Matrix Multiplication

In [ ]:
batch_size=64; img_dim=224
inputs = np.random.randn(batch_size, img_dim * img_dim).astype(np.float32)
weights = np.random.randn(img_dim * img_dim, 1000).astype(np.float32)
inference.shape = np.dot(inputs, weights)
throughput = batch_size / duration

### `MEMORY LATENCY (Random Access) Constructor`
* via Sparse Embedding Lookups (gathering specific rows from a massive table)

In [ ]:
num_users=100_000; num_lookups=1000
embedding_table = np.random.rand(num_users, 128).astype(np.float32)
indices = np.random.randint(0, num_users, size=num_lookups)
latency_us = duration/num_lookups * 1e6     # find duration for embedding_table[indices]

### `LOGIC/CONTROL CPU BOUND Constructor`
* via Control flow & CPU logic (lots of conditional branching) 

In [ ]:
num_moves=5000; board_state = np.zeros((10, 10)); moves_made = 0
for _ in range(num_moves):
    x, y = np.random.randint(0, 10, 2)
    if board_state[x, y] == 0:
        board_state[x, y] = 1
    else:
        board_state[x, y] = 0
    moves_made += 1
moves_per_second = num_moves / duration     # calculate duration for the loop

### `Compute Efficient Frontier`

<img src="images/neuralscalinglaw1.png" width="1311">
<!-- ![image.png](images/neuralscalinglaw1.png) -->

### `Manifold Hypothesis`

<img src="images/manifoldhypothesis1.png" width="1511">
<!-- ![image.png](images/manifoldhypothesis1.png) -->

<img src="images/manifoldhypothesis2.png">

# <span style="background-color:green; color:black">TRANSFORMER</span>

### `-----value class`

<img src="images/autograd1.png" width="1311">

In [ ]:
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')                                                                                   # Python optimization for memory usage

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                                                                                                                        # scalar value of this node calculated during forward pass
        self.grad = 0                                                                                                                           # derivative of the loss w.r.t. this node, calculated in backward pass
        self._children = children                                                                                                               # children of this node in the computation graph
        self._local_grads = local_grads                                                                                                         # local derivative of this node w.r.t. its children

    def __repr__(self):
        return f"Value(data={self.data:.4f})"




    def sigmoid(self):
        # forward pass
        val = 1 / (1 + math.exp(-self.data))
        # backward pass
        local_grad = val * (1 - val)
        return Value(val, (self,), (local_grad,))

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1





    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)



        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad                                                                                                   # chain rule for differentiation

### `STATE DICTIONARY defined using Value class`

In [ ]:
n_embd = 16
n_head = 4                                                                                                                          # number of attention heads
head_dim = n_embd // n_head                                                                                                         # grammar + context + position + sentiment




block_size = 16                                                                                                                     # max number of past tokens that can be looked at (max sequence length)
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}          # initialize word token ebedding ; word position embedding ; language model head



<img src="images/tokenembedding.png" width="1711">

In [ ]:
n_layer = 2                                                                                                                         # number of layers                                                                    
                                                                    # ALL THE SUBSEQUENT MATRICES ARE TRANSPOSED & MULT. WITH INCOMING I/P, SO OUTPUT DIM. IS THE FIRST VALUE
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)                                                                        # Query: what other words should i pay attention to?
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)                                                                        # Key: here is a summary of what i mean
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)                                                                        # Value: if you pay attention to me here is a deep context
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)                                                                        # Output projection
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)                                                                    # Expand the embedding dim. to give thinking space
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)                                                                    # Compress the thinking space down to standard embd dim.



<img src="images/attentionpattern.png" width="1811">

<img src="images/valuematrix.png" width="1811">

<img src="images/unembedding.jpeg" width="1311">

In [ ]:


params = [p for mat in state_dict.values() for row in mat for p in row]                                                             # flatten params into a single list[Value]
print(f"num params: {len(params)}")



<img src="images/gpttotalparams.jpeg" width="1311">

<img src="images/contextlength.png">

In [ ]:



learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8                                                                      # Initialize Adam's memory for gradient descent using adaptive moment estimation
m = [0.0] * len(params)                                                                                                             # momentum to track direction of every single parameter
v = [0.0] * len(params)                                                                                                             # velocity/variance to track aggressiveness of every single parameter



### `MODEL ARCHITECTURE`

<img src="images/gptarchitecture.png" width="1711">

<img src="images/multiheadattention.png" width="1511">

In [ ]:
                                                                                            # Follow GPT-2, blessed among the GPTs, with minor differences: layernorm -> rmsnorm, no biases, GeLU -> ReLU
def gpt(token_id, pos_id, keys, values):

                                                                                            # GET EMBEDDING OF THE PAST TOKEN
    tok_emb = state_dict['wte'][token_id]                                                                               # token embedding
    pos_emb = state_dict['wpe'][pos_id]                                                                                 # position embedding
    x = [t + p for t, p in zip(tok_emb, pos_emb)]                                                                       # joint token and position embedding
    x = rmsnorm(x)



    for li in range(n_layer):
        x_residual = x
        x = rmsnorm(x)                                                                                                  # for skip connections


        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)                                                                                              # for KV cache containg all past tokens
        values[li].append(v)                                                                                            # for KV cache containg all past tokens



                                                                                            # Multi-head attention
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]                                                                                     # grabs a 4-numbered slice of the past token's query 'q'
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]                                                               # grabs a 4-numbered slice of all past tokens key
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]                                                             # grabs a 4-numbered slice of all past tokens value

            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]   # dot product of the past token's query with every past keys; divide by sqrt of dim. to keep variance stable
            attn_weights = softmax(attn_logits)                                                                         # [0.1, 0.9] = the past token is paying 10% attention to its previous token and 90% attention to itself

            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]             # multiply the attn_weights with value vetor and add them up, to mix the contexts of all tokens together
            x_attn.extend(head_out)                                                                                     # appends these 4-dim vector to a growing list of 16-dim vector




        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]                                                                      # new contextual meaning + original meaning




                                                                                            # 2) MLP block
        x_residual = x
        x = rmsnorm(x)

        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]                                                                                     # turns all -ve nos to 0
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])                                                                           # convert the 16-dim vector to 20-dim vector of vocab_size to give prob. for next token
    return logits


<img src="images/querykeytransformation.jpeg" width="1311">

### `-----linear, rmsnorm, softmax`

In [ ]:
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

In [ ]:
def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

In [ ]:
def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

### `TRAINING`

In [ ]:
"""
The most atomic way to train and inference a GPT in pure, dependency-free Python.
This file is the complete algorithm.
Everything else is just efficiency.
@karpathy
"""

import os                                                                                                                               # os.path.exists
import math                                                                                                                             # math.log, math.exp
import random                                                                                                                           # random.seed, random.choices, random.gauss, random.shuffle
import urllib.request
# import torch
random.seed(42)                                                                                                                         # Let there be order among chaos

                                                                                # Let there be an input dataset `docs`: list[str] of documents (e.g. a dataset of names)
os.makedirs('sample_data', exist_ok=True)
names_url = 'https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt'
urllib.request.urlretrieve(names_url, 'input.txt')
# shakespeare_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# urllib.request.urlretrieve(shakespeare_url, "sample_data/shakespeare.txt")


docs = [l.strip() for l in open('input.txt').read().strip().split('\n') if l.strip()]                                                   # list[str] of documents
random.shuffle(docs)
print(f"num docs: {len(docs)}")


# with open('sample_data/shakespeare.txt', 'r', encoding='utf-8') as f:
#     text = f.read()
# print(text[:100])
# print("length of dataset in characters: ", len(text))


<img src="images/trainingembedding.jpeg" width="1311">

In [ ]:

                                                                    # Tokenizer to tokenize the document: docs = ['Hi"] 
uchars = sorted(set(''.join(docs)))                                                                 # unique characters = ['h', 'i']; token ids = [0,1] i.e 0..n-1"""
BOS = len(uchars)                                                                                   # Beginning of Sequence (BOS) token id = 2
vocab_size = len(uchars) + 1                                                                        # total number of unique tokens, +1 is for BOS
print(f"vocab size: {vocab_size}")

# chars = sorted(list(set(text)))
# vocab_size = len(chars)
# print(''.join(chars))
# print(vocab_size)


num_steps = 1000                                                                                    # number of training steps
for step in range(num_steps):

    doc = docs[step % len(docs)]                                                                    # take first word "hi"
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]                                       # generate <BOS> h i <BOS> = [2, 0, 1, 2]
    # stoi = { ch:i for i,ch in enumerate(chars) }
    # itos = { i:ch for i,ch in enumerate(chars) }
    # encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
    # decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string
    # data = torch.tensor(encode(text), dtype=torch.long)
    
    n = min(block_size, len(tokens) - 1)                                                            # how many tokens to be predicted from the first token i.e predict h, predict i, predict <BOS> => n=3


    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]                                    # token_id is 2 (<BOS>), target_id is 0 (h)
        logits = gpt(token_id, pos_id, keys, values)                                                # given <BOS> what is the next token?
        probs = softmax(logits)

        loss_t = -probs[target_id].log()                                                            # -ve log-likelihood or cross-entropy loss
        losses.append(loss_t)

    loss = (1/n)*sum(losses)                                                                        # average of the loss of the 3 predictions i.e h, i, <BOS>
    loss.backward()                                                                                 # autograd trigger to calculate .grad for all parameters







                                                                    # Adam optimizer update: update the model parameters based on the corresponding gradients.
    lr_t = learning_rate * (1 - step / num_steps)                                                   # decay the learning rate linearly so as the training progressess we take smaller & smaller steps to avoid overshooting

    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad                                                  # update momentum
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2                                             # update velocity
        m_hat = m[i] / (1 - beta1 ** (step + 1))                                                    # since m and v started at 0, they are biased towards 0....
        v_hat = v[i] / (1 - beta2 ** (step + 1))                                                    # ....so bias correction by dividing small decimal

        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)                                          # substract a fraction of the gradient; 0.5 means sqrt.; divide by v_hat acts as speed breaker
        p.grad = 0                                                                                  # reset to zero so next training loop's gradients dont add causing math to explode

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}")



<img src="images/directionmeaning.png">

### `-----visualizing directional meanings`

In [ ]:
%%javascript
// 1. Setup the Canvas Environment
// We use a dark background to make the colors pop
element.html(`
<div id="canvas-container" style="width: 100%; height: 400px; background: #0f0f1f; position: relative; overflow: hidden; border-radius: 8px;">
    <canvas id="particleCanvas" style="display: block;"></canvas>
    <div style="position: absolute; bottom: 10px; right: 10px; color: rgba(255,255,255,0.5); font-family: sans-serif; font-size: 12px; pointer-events: none;">
        Interactive Particle Network
    </div>
</div>
`);

// 2. Initialize Canvas
const canvas = element.find("#particleCanvas")[0];
const ctx = canvas.getContext('2d');
const container = element.find("#canvas-container");

// Set canvas size to match container
let w = canvas.width = container.width();
let h = canvas.height = container.height();

// 3. Particle Configuration
const particleCount = 80;
const connectionDistance = 100;
const mouseDistance = 150;
const particles = [];

// Mouse interaction object
const mouse = { x: null, y: null };

// Track mouse movement inside the specific element
element.on('mousemove', function(e) {
    const rect = canvas.getBoundingClientRect();
    mouse.x = e.clientX - rect.left;
    mouse.y = e.clientY - rect.top;
});

element.on('mouseleave', function() {
    mouse.x = null;
    mouse.y = null;
});

// 4. Particle Class
class Particle {
    constructor() {
        this.x = Math.random() * w;
        this.y = Math.random() * h;
        this.vx = (Math.random() - 0.5) * 1.5; // Random horizontal velocity
        this.vy = (Math.random() - 0.5) * 1.5; // Random vertical velocity
        this.size = Math.random() * 2 + 1;
        // Random HSL Color (Pastel/Neon vibes)
        this.color = `hsl(${Math.random() * 360}, 70%, 60%)`;
    }

    update() {
        // Move
        this.x += this.vx;
        this.y += this.vy;

        // Bounce off walls
        if (this.x < 0 || this.x > w) this.vx *= -1;
        if (this.y < 0 || this.y > h) this.vy *= -1;

        // Mouse Interaction: Flee from mouse
        if (mouse.x != null) {
            let dx = mouse.x - this.x;
            let dy = mouse.y - this.y;
            let distance = Math.sqrt(dx * dx + dy * dy);
            
            if (distance < mouseDistance) {
                const forceDirectionX = dx / distance;
                const forceDirectionY = dy / distance;
                const force = (mouseDistance - distance) / mouseDistance;
                // Push particle away
                this.vx -= forceDirectionX * force * 0.5;
                this.vy -= forceDirectionY * force * 0.5;
            }
        }
    }

    draw() {
        ctx.beginPath();
        ctx.arc(this.x, this.y, this.size, 0, Math.PI * 2);
        ctx.fillStyle = this.color;
        ctx.fill();
    }
}

// Create initial particles
for (let i = 0; i < particleCount; i++) {
    particles.push(new Particle());
}

// 5. The Animation Loop
function animate() {
    // Check if canvas is still in DOM (stops loop if cell is deleted/re-run)
    if (!document.body.contains(canvas)) return;

    // Clear screen (with slight fade effect for trails? No, lets keep it clean)
    ctx.clearRect(0, 0, w, h);

    // Update and Draw Particles
    for (let i = 0; i < particles.length; i++) {
        particles[i].update();
        particles[i].draw();

        // Draw connections (The "Network" effect)
        for (let j = i; j < particles.length; j++) {
            let dx = particles[i].x - particles[j].x;
            let dy = particles[i].y - particles[j].y;
            let distance = Math.sqrt(dx * dx + dy * dy);

            if (distance < connectionDistance) {
                ctx.beginPath();
                // Line opacity based on distance (closer = brighter)
                let opacity = 1 - (distance / connectionDistance);
                ctx.strokeStyle = `rgba(255, 255, 255, ${opacity * 0.2})`;
                ctx.lineWidth = 1;
                ctx.moveTo(particles[i].x, particles[i].y);
                ctx.lineTo(particles[j].x, particles[j].y);
                ctx.stroke();
            }
        }
    }
    
    requestAnimationFrame(animate);
}

// Handle window resize (optional robustness)
window.addEventListener('resize', function() {
    if (document.body.contains(canvas)) {
        w = canvas.width = container.width();
        h = canvas.height = container.height();
    }
});

animate();

### `INFERENCE`

<img src="images/softmaxwithtemperature.png">

In [ ]:

                                                            # Inference: may the model babble back to us
temperature = 0.5                                                                           # in (0, 1], control the "creativity" of generated text, low to high
print("\n--- inference (new, hallucinated names) ---")


for sample_idx in range(20):                                                                # generate 20 new words: <BOS>word<BOS>
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]               # clear cache for each word
    token_id = BOS                                                                          # set the starting token_id to <BOS>
    sample = []                                                                             # store the tokens
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)                                        # token_id of <BOS> & pos_id=0; returns a list of probs. (length of list = vocab_size)
        probs = softmax([l / temperature for l in logits])                                  # temp. incr. the dist. b/w nos. so softmax converts large nos to even larger & small nos. close to zero
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]    # a roulette wheel of tokens based on their probs.; the reason why LLMs are non deterministic: why diff. answers everytime; reason for neural scaling laws
        if token_id == BOS:
            break
        sample.append(uchars[token_id])                                                     # append 1st token after <BOS>; for next loop token_id is this token's id; KV cache has <BOS>'s for next loop
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")

<img src="images/inferenceembedding.jpeg" width="1011">

### `-----scope of compression`

In [ ]:


                                                        # print the dimensions of the matrices in state_dict
print("\n--- state_dict (dimensions) ---")
for key, value in state_dict.items():
    print(key, (len(value), len(value[0])))



                                                        # display the matrices in state_dict in human readable format
print("\n--- state_dict (human readable) ---")
for key, value in state_dict.items():
    print(f"\n{key}:")
    for row in value:
        print("  " + " ".join(f"{v.data:7.4f}" for v in row))




# <span style="background-color:orange; color:black">Train on Kaggle with WandB</span>

In [ ]:
%%writefile kaggleformer.py
import os
from kaggle_secrets import UserSecretsClient
import math
import random
import time
import datetime
import json
import wandb

# --- Configuration ---
config = {
    "n_embd": 16,
    "n_head": 4,
    "block_size": 16,
    "n_layer": 2,
    "learning_rate": 0.01,
    "num_steps": 100000,
    "batch_size": 1,
    "max_runtime_hours": 12,
}

# Initialize WandB
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api)

run = wandb.init(project="playground-gpt", name=f"run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}", config=config)

random.seed(42)

# --- Data Loading ---
if not os.path.exists('input.txt'):
    import urllib.request
    print("Downloading input.txt...")
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')

docs = [l.strip() for l in open('input.txt').read().strip().split('\n') if l.strip()]
random.shuffle(docs)
print(f"num docs: {len(docs)}")

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1
print(f"vocab size: {vocab_size}")

# --- Model Definition ---
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __repr__(self):
        return f"Value(data={self.data:.4f})"

    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))
    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

n_embd = config['n_embd']
n_head = config['n_head']
head_dim = n_embd // n_head
block_size = config['block_size']
n_layer = config['n_layer']

matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}

for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)

params = [p for mat in state_dict.values() for row in mat for p in row]
print(f"num params: {len(params)}")

# --- Utils ---
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id]
    pos_emb = state_dict['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]
    x = rmsnorm(x)

    for li in range(n_layer):
        x_residual = x
        x = rmsnorm(x)

        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)

        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]

            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)

            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)

        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]

        x_residual = x
        x = rmsnorm(x)

        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

# --- Training Loop --- 
learning_rate, beta1, beta2, eps_adam = config['learning_rate'], 0.85, 0.99, 1e-8
m = [0.0] * len(params)
v = [0.0] * len(params)

num_steps = config['num_steps']
start_time = time.time()
max_duration = config['max_runtime_hours'] * 3600

print("Starting training loop...")

for step in range(num_steps):
    # Check for timeout
    if time.time() - start_time > max_duration:
        print(f"Reached maximum runtime of {config['max_runtime_hours']} hours. Stopping.")
        break

    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)

    loss = (1/n)*sum(losses)
    loss.backward()

    lr_t = learning_rate * (1 - step / num_steps)

    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    # Log to WandB
    wandb.log({"loss": loss.data, "step": step, "lr": lr_t})

    if step % 100 == 0:
        print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}")

print("Training completed.")

# --- Save Model ---
print("Saving model weights to model.json...")
# Convert state_dict (Value objects) to list of floats
raw_state_dict = {}
for k, v in state_dict.items():
    # v is a list of lists of Values
    raw_state_dict[k] = [[val.data for val in row] for row in v]

with open("model.json", "w") as f:
    json.dump(raw_state_dict, f)

# Log to WandB as Artifact
artifact = wandb.Artifact("kaggleformer-weights", type="model")
artifact.add_file("model.json")
run.log_artifact(artifact)

wandb.finish()


Writing kaggleformer.py


: 

### `-----retrieve trained model from wandb`

In [ ]:
import json
import math
import random
import os
import wandb
from dotenv import load_dotenv

load_dotenv()

# --- Configuration (Must match training) ---
config = {
    "n_embd": 16,
    "n_head": 4,
    "block_size": 16,
    "n_layer": 2,
    "vocab_size": 28, # a-z, space, special chars... rough estimate, logic handles it dynamically if needed
}

# --- Model Definition (Copied from playground.py/kaggleformer.py) ---
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __repr__(self):
        return f"Value(data={self.data:.4f})"

    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))
    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

# Initialize wandb
# Initialize wandb
wandb_api = os.getenv("WANDB_API_KEY")
if wandb_api:
    wandb.login(key=wandb_api)
else:
    print("Warning: WANDB_API_KEY not found in environment.")

# --- Load Function ---
def load_model(run_path=None, local_file="model.json"):
    """
    Loads model weights from WandB (or local file) into the model structure.
    Args:
        run_path: WandB run path (e.g., 'username/project/run_id'). If None, tries local file.
        local_file: Path to local JSON file.
    """
    if run_path:
        print(f"Downloading model from WandB run: {run_path}...")
        api = wandb.Api()
        run = api.run(run_path)
        # Assuming the artifact is named 'kaggleformer-weights' or similar, 
        # but simpler to just loop through logged artifacts or files.
        # For this example, we'll try to find 'model.json' in the run's files.
        file_obj = run.file("model.json")
        file_obj.download(replace=True)
        print("Downloaded model.json")
    
    if not os.path.exists(local_file):
        raise FileNotFoundError(f"Could not find model file: {local_file}")

    print(f"Loading weights from {local_file}...")
    with open(local_file, "r") as f:
        raw_state_dict = json.load(f)

    # Initialize Model Structure based on config + loaded weights dimensions
    # We need to infer vocab_size from the loaded weights if possible, 
    # or rely on config. 
    # Let's check wte size.
    if 'wte' in raw_state_dict:
        vocab_size = len(raw_state_dict['wte'])
        n_embd = len(raw_state_dict['wte'][0])
        print(f"Inferred vocab_size={vocab_size}, n_embd={n_embd}")
    else:
        raise ValueError("Invalid model file: missing 'wte'")

    # Re-create state_dict with Value objects initialized with loaded data
    state_dict = {}
    for k, v in raw_state_dict.items():
        # v is list of list of floats
        # state_dict[k] needs to be list of list of Values
        state_dict[k] = [[Value(data) for data in row] for row in v]
    
    return state_dict, vocab_size

# --- Inference Demo ---
def generate(state_dict, vocab_size, n_tokens=20, temp=0.5):
    # Needs gpt function - defined inside to close over state_dict or passed explicitly
    # To keep it clean, let's redefine gpt to take state_dict
    
    n_layer = config['n_layer']
    n_head = config['n_head']
    n_embd = config['n_embd']
    head_dim = n_embd // n_head
    block_size = config['block_size']

    # Redefine gpt here to use the passed state_dict
    def gpt_forward(token_id, pos_id, keys, values):
        tok_emb = state_dict['wte'][token_id]
        pos_emb = state_dict['wpe'][pos_id]
        x = [t + p for t, p in zip(tok_emb, pos_emb)]
        x = rmsnorm(x)

        for li in range(n_layer):
            x_residual = x
            x = rmsnorm(x)

            q = linear(x, state_dict[f'layer{li}.attn_wq'])
            k = linear(x, state_dict[f'layer{li}.attn_wk'])
            v = linear(x, state_dict[f'layer{li}.attn_wv'])
            keys[li].append(k)
            values[li].append(v)

            x_attn = []
            for h in range(n_head):
                hs = h * head_dim
                q_h = q[hs:hs+head_dim]
                k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
                v_h = [vi[hs:hs+head_dim] for vi in values[li]]

                attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
                attn_weights = softmax(attn_logits)

                head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
                x_attn.extend(head_out)

            x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
            x = [a + b for a, b in zip(x, x_residual)]

            x_residual = x
            x = rmsnorm(x)

            x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
            x = [xi.relu() for xi in x]
            x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
            x = [a + b for a, b in zip(x, x_residual)]

        logits = linear(x, state_dict['lm_head'])
        return logits

    # Assuming chars map is standard 28 chars (a-z + . + space?) 
    # Ideally should save uchars to model.json too.
    # For now, let's just use indices or try to reconstruct.
    # If using standard makemore names.txt:
    # uchars = sorted(set(''.join(open('input.txt').read().split()))) 
    # But we don't have input.txt.
    # Let's just output indices if uchars not available.
    
    print("\n--- Generating ---")
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    # Assuming BOS is vocab_size - 1 (usually appended at end)
    BOS = vocab_size - 1 
    token_id = BOS
    
    out = []
    for _ in range(n_tokens):
        logits = gpt_forward(token_id, 0, keys, values) # pos_id simplified
        probs = softmax([l.data / temp for l in logits])
        
        # Sample
        token_id = random.choices(range(vocab_size), weights=probs)[0]
        if token_id == BOS:
            break
        out.append(token_id)
    
    print(f"Generated tokens: {out}")
    # Decode if possible
    # uchars = ...
    # print("".join([uchars[i] for i in out]))

if __name__ == "__main__":
    # Example usage:
    # state_dict, vocab_size = load_model(run_path="myuser/playground-gpt/runid")
    # OR local:
    try:
        state_dict, vocab_size = load_model(local_file="model.json")
        generate(state_dict, vocab_size)
        print("Model loaded successfully!")
    except Exception as e:
        print(f"Could not load model: {e}")
        print("Ensure 'model.json' is present or provide WandB run path.")
